In [1]:
import pandas as pd
from collections import Counter
import os
import numpy as np
from tqdm import tqdm
import pickle
from sklearn.model_selection import StratifiedKFold
from collections import Counter
import gcsfs
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy.stats import mannwhitneyu

bucket = os.getenv("WORKSPACE_BUCKET")
# print(bucket)

def write_df_to_bucket(df, filename: str):
    # save mt directly to bucket
    df.to_csv(f"{bucket}/data/{filename}.csv", index = None)
def read_df_from_bucket(filename: str):
    # save mt directly to bucket
    return pd.read_csv(f"{bucket}/data/{filename}.csv")

fs = gcsfs.GCSFileSystem()
def save_dict(data_dict, filename: str):
    full_path = f"{bucket}/data/{filename}.pickle"
    with fs.open(full_path, "wb") as h:
        pickle.dump(data_dict, h)
def load_dict(filename: str):
    full_path = f"{bucket}/data/{filename}.pickle"
    with fs.open(full_path, "rb") as h:
        data_dict = pickle.load(h)
    return data_dict
def chunk_list(lst, chunk_size):
    return [lst[i:i + chunk_size] for i in range(0, len(lst), chunk_size)]

def count_label(data_dict):
    la = []
    for k, v in data_dict.items():
        la.append(v['label'])
    print(Counter(la))

# Baseline model

In [10]:
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, classification_report
from sklearn.metrics import average_precision_score
from tqdm import tqdm
import xgboost as xgb

class Baseline:
    def __init__(self, data_dict):
        self.data = data_dict.copy()
        self.X_train = []
        self.y_train = []
        self.X_test = []
        self.y_test = []
        self.test_ids = []
        self.features = []
#         self.feature_dict = {'concept_'+str(cid):0 for cid in fea_lst}
    def train_test(self):
        X_train_dicts, y_train = [], []
        X_test_dicts, y_test = [], []

        for patient_id, record in tqdm(self.data.items()):
#             for cid in record['seq']:
#                 self.feature_dict['concept_'+str(cid)] = 1
            feature_dict = {'concept_' + str(cid): 1 for cid in record['seq']}
            label = record['label']
            
            if record['split'] == 'train':
                X_train_dicts.append(feature_dict)
                self.y_train.append(label)
            elif record['split'] == 'test':
                X_test_dicts.append(feature_dict)
                self.y_test.append(label)
                self.test_ids.append(patient_id) 

        vec = DictVectorizer(sparse=True)
        self.X_train = vec.fit_transform(X_train_dicts)
        self.X_test = vec.transform(X_test_dicts)
        self.features = vec.get_feature_names_out()

    def eval_xgboost(self):
        model = xgb.XGBClassifier(
            objective='binary:logistic',
            eval_metric='auc',
            use_label_encoder=False,
            n_estimators=100,
            max_depth=6,
            learning_rate=0.1,
            subsample=0.8,
            colsample_bytree=0.8,
            n_jobs=-1
        )
        model.fit(self.X_train, self.y_train)
        y_proba = model.predict_proba(self.X_test)[:, 1]
        auc = roc_auc_score(self.y_test, y_proba)
        auprc = average_precision_score(self.y_test, y_proba)
        print(f"[XGBoost] AUC: {auc:.4f} | AUPRC: {auprc:.4f}")
        
        xgboost_df = pd.DataFrame()
        xgboost_df['person_id'] = self.test_ids
        xgboost_df['prob'] = y_proba
        xgboost_df['label'] = self.y_test

        return model, xgboost_df

In [28]:
def make_new_dict(data_dict, year = 5, excl_ids = []):
    new_dict = {}
    for i in data_dict:
        new_time = []
        new_seq = []

        for t, s in zip(data_dict[i]['time'], data_dict[i]['seq']):
            t3 = t - (year-1)  # shift from 1yr to 3yr anchor

            if t3 > 0 and s not in set(excl_ids):   # retain only events before 3yr anchor
                new_time.append(t3)
                new_seq.append(s)
        if len(new_seq)>=5:
            new_dict[i] = {}
            new_dict[i]['seq'] = new_seq
            new_dict[i]['time'] = new_time
            new_dict[i]['label'] = data_dict[i]['label']
            new_dict[i]['split'] = data_dict[i]['split']
            
    return new_dict

In [14]:
concept = read_df_from_bucket('concept')
concept_dict = dict(zip(concept['concept_id'], concept['concept_name']))

In [22]:
excl_names = {}
excl_names['breast'] = ['Carcinoma in situ of breast', 'Neoplasm of uncertain behavior of breast',
                        'Abnormal findings on diagnostic imaging of breast',
                        'Microcalcifications of the breast',
                        'Mammography abnormal', 'Acquired absence of breast', 'Fibrocystic disease of breast']

excl_ids = {}
for cancer_type in cancer_types:
    excl_ids[cancer_type] = []
    for i in excl_names['breast']:
        excl_ids[cancer_type].extend(concept[concept['concept_name']==i]['concept_id'].tolist())

In [46]:
excl_ids_sql = ",".join(str(x) for x in excl_ids["breast"])
query = f"""
SELECT ca.descendant_concept_id
FROM `{os.environ["WORKSPACE_CDR"]}.concept_ancestor` ca
WHERE ca.ancestor_concept_id IN ({excl_ids_sql})
"""

excl_all_ids = pd.read_gbq(
    query,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
    progress_bar_type="tqdm_notebook")

/tmp/ipykernel_593/1225154142.py:8: FutureWarning: read_gbq is deprecated and will be removed in a future version. Please use pandas_gbq.read_gbq instead: https://pandas-gbq.readthedocs.io/en/latest/api.html#pandas_gbq.read_gbq
  excl_all_ids = pd.read_gbq(


Downloading:   0%|          |

In [47]:
len(excl_all_ids)

256

In [49]:
excl_ids[cancer_type] = excl_all_ids['descendant_concept_id'].tolist()

In [2]:
cancer_types = ['prostate','stomach','ovarian','crc', 'lung','liver','pancreas']
excl_names['prostate'] = ['Raised prostate specific antigen','Urine cytology abnormal','Blood in urine']
excl_names['crc'] = ['Gastrointestinal hemorrhage','Disorder of anal region']
excl_names['lung'] = ['Benign neoplasm of bronchus and lung','Preoperative state','Complication of medical care']
excl_names['ovarian'] = ["Intra-abdominal and pelvic swelling, mass and lump", "Cyst of ovary"]
excl_names['pancreas'] = ["Cyst of pancreas","Cyst and pseudocyst of pancreas",
                         "Benign neoplasm of pancreas","Acute pancreatitis","Complication of procedure",
                         "Postoperative infection"]
excl_names['liver'] = ['Transplanted liver present']
excl_names['stomach'] = ["Iron deficiency anemia", "Anal and rectal polyp", "Benign neoplasm of rectum and anal canal"]

for cancer_type in tqdm(cancer_types):
    excl_ids[cancer_type] = []
    for i in excl_names[cancer_type]:
        excl_ids[cancer_type].extend(concept[concept['concept_name']==i]['concept_id'].tolist())
        
for cancer_type in tqdm(cancer_types): 
    excl_ids_sql = ",".join(str(x) for x in excl_ids[cancer_type])
    query = f"""
    SELECT ca.descendant_concept_id
    FROM `{os.environ["WORKSPACE_CDR"]}.concept_ancestor` ca
    WHERE ca.ancestor_concept_id IN ({excl_ids_sql})
    """

    excl_all_ids = pd.read_gbq(
        query,
        dialect="standard",
        use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
        progress_bar_type="tqdm_notebook")

    print(cancer_type, len(excl_ids[cancer_type]))
    excl_ids[cancer_type] = excl_all_ids['descendant_concept_id'].tolist()
    print(cancer_type, len(excl_ids[cancer_type]))

In [3]:
import shap

cancer_types = ['prostate','stomach','ovarian','crc', 'lung','liver','pancreas']
# cancer_types = ['breast']
year_before = 3
for cancer_type in cancer_types:
    print(cancer_type)
    explainer, shap_values, auroc_df= {}, {}, {}
    for fold in range(5):
        if cancer_type in ['crc', 'lung','liver']:
            seq_data = load_dict(f'seq_data_{cancer_type}_condition_no_empty'+str(fold)) #crc, lung, liver
        elif cancer_type in ['pancreas']:
            seq_data = load_dict(f'seq_data_condition'+str(fold)) #pancreas
        else:
            seq_data = load_dict(f'seq_data_{cancer_type}_condition'+str(fold)) #breast, prostate, stomoch
        ctrl_dict = load_dict('seq_data_condition_remaining_ctrl'+str(fold))
        seq_data.update(ctrl_dict)
        
        print(len(seq_data))
        seq_data = make_new_dict(seq_data, year = year_before, excl_ids = excl_ids[cancer_type])
        print(len(seq_data))
        
        baseline = Baseline(seq_data)
        baseline.train_test()
        fea_names = [concept_dict[int(i.split('_')[1])] for i in baseline.features]

        model, res_df = baseline.eval_xgboost()
        X = baseline.X_test.toarray()
        fea_names = baseline.features
        fea_names = [concept_dict[int(i.split('_')[1])] for i in fea_names]
        explainer[fold] = shap.Explainer(model, feature_names=fea_names)
        shap_values[fold] = explainer[fold](X)
        auroc_df[fold] = res_df

    fea_dict = {}
    for fold in tqdm(range(5)):
        fea_dict[fold] = {}

        shap_df = pd.DataFrame(shap_values[fold].values, columns=shap_values[fold].feature_names)

        # Sum SHAP values across samples
        score = shap_df.sum()

        # Filter out zero scores
        score = score[score != 0]

        # Sort by descending score
        sorted_score = score.sort_values(ascending=False)

        # Assign rank (1 = most important)
        for rank, (feature, value) in enumerate(sorted_score.items(), start=1):
            fea_dict[fold][feature] = rank

    common_fea = set(list(fea_dict[0].keys()))
    for fold in range(5):
        common_fea = common_fea&set(list(fea_dict[fold].keys()))

    common_fea_dict = {}
    for i in common_fea:
        avg = []
        for fold in range(5):
            avg.append(fea_dict[fold][i])
        avg = np.mean(avg)
        common_fea_dict[i] = avg
    sorted_common_fea = dict(sorted(common_fea_dict.items(), key=lambda item: item[1]))
    save_dict(sorted_common_fea, f'shap_{cancer_type}_no_WGS_restriction_{year_before}year_excl')

    for fold in range(5):
        write_df_to_bucket(auroc_df[fold], f'xgboost_df_{cancer_type}_all_{year_before}year_excl'+str(fold))

In [4]:
import seaborn as sns
cancer_types = ['breast', 'prostate','stomach','ovarian','crc', 'lung','liver','pancreas']

num = 20

for cancer_type in cancer_types:
    fea = load_dict(f'shap_{cancer_type}_no_WGS_restriction_{year_before}year_excl')
    df = pd.DataFrame({'feature':list(fea.keys())[0:num], 'avg rank':list(fea.values())[0:num]})
    features = list(fea.keys())[0:num] # Top 20 features

    plt.figure(figsize=(8, 5))
    sns.barplot(x='avg rank', y='feature', data=df, palette='viridis')
    plt.xlabel('Avg. Rank', fontsize=12)
    plt.ylabel('')
    plt.xlim([0,130])
    if cancer_type=='crc':
        cancer_type = 'colorectal'
    plt.title(cancer_type, fontsize = 15)
    plt.tight_layout()